<div style="border:2px solid #4CAF50; padding:16px; border-radius:10px; background-color:#f9fff9;">
<h1 style="color:#2e7d32; margin-top:0; text-align:center; font-weight:bold;">
CatBoost  for Irrigation Need Prediction</h1>
<h3 style="color:#66bb6a; margin-top:-8px; text-align:center;">Playground Series S6E4</h3></div>

# <span style="color:#2e7d32; text-align:center; display:block; font-weight:bold;">Setup</span>

In [1]:
import warnings
warnings.filterwarnings("ignore")
import gc
from itertools import combinations
from pathlib import Path
import numpy as np
import pandas as pd
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)
from catboost import CatBoostClassifier, Pool
from sklearn.metrics import *
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import TargetEncoder
from sklearn.utils.class_weight import compute_sample_weight

# ANSI colour codes
GREEN = "\033[92m"
YELLOW = "\033[93m"
CYAN = "\033[36m"
BOLD = "\033[1m"
RESET = "\033[0m"

def print_header(text):
    print(f"\n{BOLD}{CYAN}{'━'*45}{RESET}")
    print(f"{BOLD}{CYAN}{text.center(80)}{RESET}")
    print(f"{BOLD}{CYAN}{'━'*450}{RESET}\n")

def print_success(text):
    print(f"{BOLD}{GREEN}✓ {text}{RESET}")

# <span style="color:#2e7d32; text-align:center; display:block; font-weight:bold;">Configuration</span>

In [2]:
TARGET = "Irrigation_Need"

# Label ordering: internal (model) vs public (submission)
INTERNAL_ORDER = ["Low", "Medium", "High"]
PUBLIC_ORDER   = ["High", "Low", "Medium"]

# Original numeric and categorical columns
NUMS = [
    "Soil_pH", "Soil_Moisture", "Organic_Carbon", "Electrical_Conductivity",
    "Temperature_C", "Humidity", "Rainfall_mm", "Sunlight_Hours",
    "Wind_Speed_kmh", "Field_Area_hectare", "Previous_Irrigation_mm",
]

CATS = [
    "Soil_Type", "Crop_Type", "Crop_Growth_Stage", "Season",
    "Irrigation_Type", "Water_Source", "Mulching_Used", "Region",
]

ALL_SOURCE = NUMS + CATS   # For pairwise interactions

# Training parameters
ORIG_ROW_WEIGHT = 0.35      # Down‑weight rows from original dataset
N_FOLDS = 5
SEED = 42

# Class weights 
CLASS_WEIGHTS = {0: 1.1626, 1: 1.3486, 2: 2.6307}

# <span style="color:#2e7d32; text-align:center; display:block; font-weight:bold;">Data Loading</span>

In [3]:
DATA_DIR = Path("/kaggle/input/competitions/playground-series-s6e4")
ORIG_DIR = Path("/kaggle/input/datasets/miadul/irrigation-water-requirement-prediction-dataset")

train = pd.read_csv(DATA_DIR / "train.csv").set_index("id")
test  = pd.read_csv(DATA_DIR / "test.csv").set_index("id")
sample_sub = pd.read_csv(DATA_DIR / "sample_submission.csv")

# Load original dataset 
orig = pd.read_csv(next(ORIG_DIR.rglob("*.csv")))
if "id" in orig.columns:
    orig = orig.drop(columns=["id"])
print_success(f"Original dataset loaded")

print(f"Original dataset shape : {orig.shape}")
print(f"Train dataset shape    : {train.shape}")
print(f"Test dataset shape     : {test.shape}")

# Encode target labels to integers (0=Low, 1=Medium, 2=High)
target_map = {l: i for i, l in enumerate(INTERNAL_ORDER)}
train[TARGET] = train[TARGET].map(target_map)
orig[TARGET]  = orig[TARGET].map(target_map)

y_internal = train[TARGET].to_numpy(dtype=np.int64)

# Public label mapping for submission
public_map = {l: i for i, l in enumerate(PUBLIC_ORDER)}
i2p = {target_map[l]: public_map[l] for l in INTERNAL_ORDER}
y_public = np.array([i2p[v] for v in y_internal], dtype=np.int64)
print_success("Encoding completed successfully.")

✓ Original dataset loaded
Original dataset shape : (10000, 20)
Train dataset shape    : (630000, 20)
Test dataset shape     : (270000, 19)
✓ Encoding completed successfully.


# <span style="color:#2e7d32; text-align:center; display:block; font-weight:bold;">Feature Engineering</span>
<div style="font-family:Arial,sans-serif;line-height:1.5;">
<p>We add:</p>
<ul>
<li>Domain-specific features (moisture ratios, stress indices, squares)</li>
<li>Evapotranspiration features (ETo, Kc, ETc, water balance)</li>
<li>Digit features (from XGB pipeline)</li>
<li>Pairwise interaction features (with cardinality filtering)</li>
<li>Original dataset prior means for each categorical/numeric column</li>
</ul></div>

In [4]:
# 1 Domain features (ratios, stress indices, squares)
def add_domain_features(*dfs):
    for d in dfs:
        d["moist_rain"]    = d["Soil_Moisture"] / (d["Rainfall_mm"] + 1)
        d["moist_temp"]    = d["Soil_Moisture"] / (d["Temperature_C"] + 1)
        d["moist_wind"]    = d["Soil_Moisture"] / (d["Wind_Speed_kmh"] + 1)
        d["ET_proxy"]      = (d["Temperature_C"] * d["Wind_Speed_kmh"] * d["Sunlight_Hours"]) / (d["Humidity"] + 1)
        d["heat_stress"]   = d["Temperature_C"] * d["Sunlight_Hours"]
        d["drying_force"]  = d["Wind_Speed_kmh"] * d["Temperature_C"] / (d["Humidity"] + 1)
        d["water_supply"]  = d["Rainfall_mm"] + d["Previous_Irrigation_mm"]
        d["water_deficit"] = d["Soil_Moisture"] - d["water_supply"] * 0.1
        d["soil_quality"]  = d["Organic_Carbon"] / (d["Electrical_Conductivity"] + 0.1)
        d["moist_x_wind"]  = d["Soil_Moisture"] * d["Wind_Speed_kmh"]
        d["moist_x_temp"]  = d["Soil_Moisture"] * d["Temperature_C"]
        d["wind_x_temp"]   = d["Wind_Speed_kmh"] * d["Temperature_C"]
        d["moisture_sq"]   = d["Soil_Moisture"] ** 2
        d["wind_sq"]       = d["Wind_Speed_kmh"] ** 2
        d["temp_sq"]       = d["Temperature_C"] ** 2

print("Adding domain features...")
add_domain_features(train, test, orig)

# 2 Evapotranspiration features (ETo, Kc, ETc, water balance)
def add_evapotranspiration_features(*dfs):
    # Reference evapotranspiration (simplified Hargreaves)
    for d in dfs:
        T_mean = d["Temperature_C"]
        T_range = 10.0
        Ra = d["Sunlight_Hours"] / 12.0 * 15.0
        d["ETo"] = 0.0023 * (T_mean + 17.8) * np.sqrt(T_range) * Ra
        d["ETo"] = d["ETo"].clip(0, 10)

    crop_kc = {
        "Wheat": {"Initial": 0.4, "Mid": 1.1, "Late": 0.3},
        "Rice": {"Initial": 1.05, "Mid": 1.2, "Late": 0.9},
        "Maize": {"Initial": 0.3, "Mid": 1.2, "Late": 0.6},
        "Soybean": {"Initial": 0.4, "Mid": 1.15, "Late": 0.5},
        "Cotton": {"Initial": 0.4, "Mid": 1.15, "Late": 0.7},
        "Sorghum": {"Initial": 0.3, "Mid": 1.0, "Late": 0.5},
        "Barley": {"Initial": 0.3, "Mid": 1.1, "Late": 0.4},
        "Sunflower": {"Initial": 0.3, "Mid": 1.1, "Late": 0.5},
    }
    stage_map = {"Initial": 0, "Mid": 1, "Late": 2}

    for d in dfs:
        d["Kc"] = 0.8
        for crop, stages in crop_kc.items():
            mask = (d["Crop_Type"] == crop)
            if mask.any():
                stage = d.loc[mask, "Crop_Growth_Stage"].map(stage_map).fillna(1).astype(int)
                kc_vals = np.array([stages["Initial"], stages["Mid"], stages["Late"]])
                d.loc[mask, "Kc"] = kc_vals[stage]
        d["ETc"] = d["Kc"] * d["ETo"]
        d["water_balance"] = d["Soil_Moisture"] - d["ETc"]

print("Adding evapotranspiration features...")
add_evapotranspiration_features(train, test, orig)

# 3 Joint factorisation of categorical columns (consistent codes across train/test/orig)
def factorize_joint(tr, te, og):
    for col in CATS:
        combined = pd.concat([tr[col], te[col], og[col]], ignore_index=True)
        codes, _ = pd.factorize(combined)
        tr[col] = pd.Series(codes[:len(tr)], index=tr.index).astype("int32").astype("category")
        te[col] = pd.Series(codes[len(tr):len(tr)+len(te)], index=te.index).astype("int32").astype("category")
        og[col] = pd.Series(codes[len(tr)+len(te):], index=og.index).astype("int32").astype("category")

print("Factorizing categoricals...")
factorize_joint(train, test, orig)

# 4 Pairwise interaction features (all combos of ALL_SOURCE, cardinality filtering)
def build_pair_columns(tr, te, og):
    te_cols = []
    total = len(tr) + len(te) + len(og)
    for left, right in combinations(ALL_SOURCE, 2):
        name = f"{left}__{right}"
        tr[name] = tr[left].astype(str) + "_" + tr[right].astype(str)
        te[name] = te[left].astype(str) + "_" + te[right].astype(str)
        og[name] = og[left].astype(str) + "_" + og[right].astype(str)

        combined = pd.concat([tr[name], te[name], og[name]], ignore_index=True)
        codes, _ = pd.factorize(combined)

        if pd.Series(codes).nunique() > total // 2:
            tr.drop(columns=[name], inplace=True)
            te.drop(columns=[name], inplace=True)
            og.drop(columns=[name], inplace=True)
            continue

        tr[name] = codes[:len(tr)].astype("int32")
        te[name] = codes[len(tr):len(tr)+len(te)].astype("int32")
        og[name] = codes[len(tr)+len(te):].astype("int32")
        te_cols.append(name)
    return te_cols

print("Building pairwise combos...")
te_columns = build_pair_columns(train, test, orig)
print_success(f"Kept {len(te_columns)} pairwise features")

# 5 Original‑dataset prior features (global target means per category)
def add_orig_priors(tr, te, og):
    created = []
    for col in CATS + NUMS:
        if og[col].nunique() == 0:
            continue
        mapping = og.groupby(col, observed=False)[TARGET].mean().astype("float32")
        name = f"TE_ORIG_{col}"
        tr[name] = tr[col].astype(object).map(mapping).fillna(0.5).astype("float32")
        te[name] = te[col].astype(object).map(mapping).fillna(0.5).astype("float32")
        og[name] = og[col].astype(object).map(mapping).fillna(0.5).astype("float32")
        created.append(name)
    return created

print("Adding original‑dataset prior features...")
add_orig_priors(train, test, orig)

feature_columns = [c for c in train.columns if c != TARGET]

print_success(" FEATURE ENGINEERING COMPLETE ")
print(f"Total features: {len(feature_columns)}")

Adding domain features...
Adding evapotranspiration features...
Factorizing categoricals...
Building pairwise combos...
✓ Kept 135 pairwise features
Adding original‑dataset prior features...
✓  FEATURE ENGINEERING COMPLETE 
Total features: 192


# <span style="color:#2e7d32; text-align:center; display:block; font-weight:bold;">Per-Fold Data Builder</span>

<div style="font-family: Arial, sans-serif; line-height:1.6;">
<p style="margin-top:10px;">
We construct a function that performs the following operations for each fold:
</p>
<ul style="padding-left:20px; margin-top:8px;">
  <li>
    <b>Data Concatenation:</b> Combine training fold with original dataset rows
  </li>
  <li>
    <b>Target Encoding:</b> Apply encoding on pairwise columns using internal 5-fold CV
  </li>
  <li>
    <b>Sample Weight Computation:</b>
    <ul style="margin-top:6px;">
      <li>Ensure balanced weighting</li>
      <li>Downweight original dataset rows</li>
      <li>Apply class-based weighting</li>
    </ul></li></ul></div>

In [5]:
def build_fold_frames(tr_fold, va_fold, te_df, og_df, feat_cols, te_cols, y_tr, seed):
    base_cols = [c for c in feat_cols if c not in te_cols]

    encoder = TargetEncoder(target_type="multiclass", cv=5, random_state=seed)

    og_y = og_df[TARGET].to_numpy(dtype=np.int64)
    combined_X = pd.concat([tr_fold[te_cols], og_df[te_cols]], ignore_index=True)
    combined_y = np.concatenate([y_tr, og_y])

    # Transform pairwise columns
    enc_tr = encoder.fit_transform(combined_X, combined_y)
    enc_va = encoder.transform(va_fold[te_cols])
    enc_te = encoder.transform(te_df[te_cols])

    # Convert to DataFrames with generic column names
    enc_tr = pd.DataFrame(enc_tr, columns=[f"te_{i}" for i in range(enc_tr.shape[1])])
    enc_va = pd.DataFrame(enc_va, columns=[f"te_{i}" for i in range(enc_va.shape[1])])
    enc_te = pd.DataFrame(enc_te, columns=[f"te_{i}" for i in range(enc_te.shape[1])])

    # Combine with base (non‑pairwise) columns
    combined_base = pd.concat([tr_fold[base_cols], og_df[base_cols]], ignore_index=True)
    x_tr = pd.concat([enc_tr, combined_base.reset_index(drop=True)], axis=1)
    x_va = pd.concat([enc_va, va_fold[base_cols].reset_index(drop=True)], axis=1)
    x_te = pd.concat([enc_te, te_df[base_cols].reset_index(drop=True)], axis=1)

    # Sample weights: start with balanced class weights
    sw = compute_sample_weight("balanced", combined_y).astype(np.float32)
    # Down‑weight rows coming from the original dataset
    sw[len(tr_fold):] *= ORIG_ROW_WEIGHT
    # Apply user‑provided class weights
    sw = sw * np.array([CLASS_WEIGHTS[y] for y in combined_y])

    # Categorical columns must be marked for CatBoost
    cat_cols = [c for c in CATS if c in x_tr.columns]

    return x_tr, combined_y, x_va, x_te, cat_cols, sw, encoder

In [6]:
# # Sample Weights for Original Training Data
# unique, counts = np.unique(y_internal, return_counts=True)
# avg_count = len(train) / len(unique)
# sample_weights_balanced = np.array([avg_count / counts[y] for y in y_internal])

# <span style="color:#2e7d32; text-align:center; display:block; font-weight:bold;">Model Training</span>

In [7]:
splitter = StratifiedKFold(N_FOLDS, shuffle=True, random_state=SEED)
folds = list(splitter.split(np.zeros(len(train)), y_internal))

oof_catboost = np.zeros((len(train), 3), dtype=np.float64)
tst_catboost_parts = []

for fold, (ti, vi) in enumerate(folds, 1):
    print(f"\n{BOLD}--- Fold {fold}/{N_FOLDS} ---{RESET}")

    tr_f, va_f = train.iloc[ti].copy(), train.iloc[vi].copy()
    y_tr = tr_f[TARGET].to_numpy(dtype=np.int64)
    y_va = va_f[TARGET].to_numpy(dtype=np.int64)

    x_tr, y_m, x_va, x_te, cat_cols, sw, _ = build_fold_frames(
        tr_f, va_f, test, orig, feature_columns, te_columns, y_tr, SEED + fold
    )

    cat_idx = [x_tr.columns.get_loc(c) for c in cat_cols]

    model = CatBoostClassifier(
        loss_function="MultiClass",
        eval_metric="TotalF1",
        iterations=3000,
        learning_rate=0.03,
        grow_policy = "SymmetricTree",
        depth=8,
        l2_leaf_reg=4.0,
        task_type="GPU",
        devices="0:1",
        random_seed=SEED + fold,
        verbose=False,
        early_stopping_rounds=200,
    )

    model.fit(
        Pool(x_tr, y_m, cat_features=cat_idx, weight=sw),
        eval_set=Pool(x_va, y_va, cat_features=cat_idx),
        use_best_model=True,
    )

    oof_catboost[vi] = model.predict_proba(Pool(x_va, cat_features=cat_idx))
    tst_catboost_parts.append(model.predict_proba(Pool(x_te, cat_features=cat_idx)).astype(np.float32))

    ba = balanced_accuracy_score(y_va, oof_catboost[vi].argmax(1))
    print(f"  {GREEN}Validation Balanced Accuracy = {ba:.5f}{RESET}")

    del model
    gc.collect()

mean_tst_catboost = np.mean(np.stack(tst_catboost_parts), axis=0)


--- Fold 1/5 ---
  Validation Balanced Accuracy = 0.97685

--- Fold 2/5 ---
  Validation Balanced Accuracy = 0.97668

--- Fold 3/5 ---
  Validation Balanced Accuracy = 0.97901

--- Fold 4/5 ---
  Validation Balanced Accuracy = 0.97726

--- Fold 5/5 ---
  Validation Balanced Accuracy = 0.97769


# <span style="color:#2e7d32; text-align:center; display:block; font-weight:bold;">Threshold Optimization</span>

In [8]:
def reorder_to_public(proba):
    src = {l: i for i, l in enumerate(INTERNAL_ORDER)}
    return proba[:, [src[l] for l in PUBLIC_ORDER]]

oof_pub = reorder_to_public(oof_catboost)

def objective(trial, proba, y_true):
    # Suggest thresholds for the three classes (public order: High, Low, Medium)
    t_high = trial.suggest_float("thresh_high", 0.1, 0.9, step=0.02)
    t_low  = trial.suggest_float("thresh_low",  0.1, 0.9, step=0.02)
    t_med  = trial.suggest_float("thresh_medium", 0.1, 0.9, step=0.02)
    thresh = np.array([t_high, t_low, t_med])
    adjusted = proba / thresh
    pred = np.argmax(adjusted, axis=1)
    return balanced_accuracy_score(y_true, pred)

study = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=SEED))
study.optimize(lambda trial: objective(trial, oof_pub, y_public), n_trials=150, show_progress_bar=False)

best_thresh = np.array([study.best_params["thresh_high"], study.best_params["thresh_low"], study.best_params["thresh_medium"]])
best_score = study.best_value

print(f"\nOptimal thresholds (public order): [{best_thresh[0]:.2f}, {best_thresh[1]:.2f}, {best_thresh[2]:.2f}]")
print(f"{GREEN}Threshold‑optimised OOF Balanced Accuracy = {best_score:.5f}{RESET}")


Optimal thresholds (public order): [0.32, 0.54, 0.72]
Threshold‑optimised OOF Balanced Accuracy = 0.97838


# <span style="color:#2e7d32; text-align:center; display:block; font-weight:bold;">Model Evaluation</span>

In [9]:
oof_adjusted = oof_pub / best_thresh
oof_preds_public = np.argmax(oof_adjusted, axis=1)

ba_final = balanced_accuracy_score(y_public, oof_preds_public)
print(f"{BOLD}{CYAN}Balanced Accuracy:{RESET} {GREEN}{ba_final:.5f}{RESET}\n")

print(f"{BOLD}{CYAN}Classification Report (per class):{RESET}")
print(f"{YELLOW}{classification_report(y_public, oof_preds_public, target_names=PUBLIC_ORDER)}{RESET}")

prec, rec, f1, _ = precision_recall_fscore_support(y_public, oof_preds_public, average=None)
print(f"\n{BOLD}{CYAN}Per-class Metrics:{RESET}")
print(f"{BOLD}Class   Precision   Recall   F1-score{RESET}")
for i, cls in enumerate(PUBLIC_ORDER):
    print(f"{GREEN}{cls:<8}{RESET} {prec[i]:.4f}     {rec[i]:.4f}   {f1[i]:.4f}")

cm = confusion_matrix(y_public, oof_preds_public)
print(f"\n{BOLD}{CYAN}Confusion Matrix (counts, rows=true, cols=predicted):{RESET}")
print("         " + "   ".join([f"{YELLOW}{c}{RESET}" for c in PUBLIC_ORDER]))
for i, cls in enumerate(PUBLIC_ORDER):
    print(f"{GREEN}{cls:<8}{RESET} {cm[i]}")

cm_norm = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]
print(f"\n{BOLD}{CYAN}Normalised Confusion Matrix (row-wise):{RESET}")
for i, cls in enumerate(PUBLIC_ORDER):
    norm_vals = "  ".join([f"{v:.2f}" for v in cm_norm[i]])
    print(f"{GREEN}{cls:<8}{RESET} {norm_vals}")

Balanced Accuracy: 0.97838

Classification Report (per class):
              precision    recall  f1-score   support

        High       0.85      0.98      0.91     21009
         Low       0.99      0.99      0.99    369917
      Medium       0.99      0.97      0.98    239074

    accuracy                           0.98    630000
   macro avg       0.94      0.98      0.96    630000
weighted avg       0.98      0.98      0.98    630000


Per-class Metrics:
Class   Precision   Recall   F1-score
High     0.8529     0.9757   0.9102
Low      0.9874     0.9939   0.9906
Medium   0.9882     0.9655   0.9767

Confusion Matrix (counts, rows=true, cols=predicted):
         High   Low   Medium
High     [20498     0   511]
Low      [     0 367671   2246]
Medium   [  3534   4703 230837]

Normalised Confusion Matrix (row-wise):
High     0.98  0.00  0.02
Low      0.00  0.99  0.01
Medium   0.01  0.02  0.97


# <span style="color:#2e7d32; text-align:center; display:block; font-weight:bold;">Final Predictions & Submission</span>

In [10]:
tst_pub = reorder_to_public(mean_tst_catboost)
adjusted_test = tst_pub / best_thresh
final_preds_public = np.argmax(adjusted_test, axis=1)

decode = np.array(PUBLIC_ORDER)
submission_labels = decode[final_preds_public]

submission = sample_sub.copy()
submission[TARGET] = submission_labels
submission.to_csv("submission.csv", index=False)

print("Submission Distribution (Public Ordering):")
print(submission[TARGET].value_counts().to_string())

print_header("✓ COMPLETED")
print_success("Submission saved as 'submission.csv'")
print_success(f"Final OOF Balanced Accuracy (public order, thresholded): {ba_final:.5f}")

Submission Distribution (Public Ordering):
Irrigation_Need
Low       159443
Medium    100207
High       10350

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
                                  ✓ COMPLETED                                   
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

✓ Submission saved as 'submission.csv'
✓ Final OOF Balanced Accuracy (public order, thresholded): 0.97838
